# 04 — Pipeline runner (Windows / VS Code)
Runs the command-line steps from inside VS Code; every step also writes `results/logs/<step>.log`.
Run the cells **one at a time, in order**. Order with the other notebooks: 00 → 04 step 1 → 01 → 02 → 04 step 2 → 03.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
%matplotlib inline
print("project root:", ROOT)

project root: <repo>


In [9]:
import subprocess, time
from pathlib import Path
LOGS = ROOT / "results" / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

def run(name, *args):
    """Run `python -m <args>` from 02_code, stream output here and to results/logs/<name>.log."""
    cmd = [sys.executable, "-u", "-m", *args]
    t0 = time.time()
    with open(LOGS / f"{name}.log", "w", encoding="utf-8") as log:
        log.write(" ".join(cmd) + "\n")
        p = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             encoding="utf-8", errors="replace")
        finished = False
        for line in p.stdout:
            print(line, end="")
            log.write(line); log.flush()
            finished = finished or line.startswith("[done] exit code 0")
        p.wait()
        log.write(f"\n[exit {p.returncode}] {time.time() - t0:.0f} s\n")
    print(f"\n[{name}] exit {p.returncode} in {time.time() - t0:.0f} s")
    # Windows sometimes crashes during interpreter shutdown (0xC0000409) AFTER all results are written.
    # A step counts as successful only if (a) it exited 0 or reported "[done] exit code 0", AND
    # (b) every run it started has complete outputs on disk (runs.csv row, curves, 240 test predictions).
    from src import verify_runs
    v = verify_runs.verify_log(LOGS / f"{name}.log")
    print(v["summary"])
    if (p.returncode != 0 and not finished) or not v["ok"]:
        raise RuntimeError(f"{name} failed -> {LOGS / (name + '.log')}")

## Step 1. Data: download (Zenodo 1188976) → metadata → fixed splits → feature cache

In [3]:
run("prepare", "src.prepare", "all")

  -> <repo>\data\raw\Audio_Speech_Actors_01-24.zip
  1 MB 1%
  21 MB 10%
  42 MB 20%
  63 MB 30%
  84 MB 40%
  105 MB 50%
  126 MB 60%
  147 MB 70%
  168 MB 80%
  189 MB 91%
Extracting Audio_Speech_Actors_01-24.zip ...
zip md5 = bc696df654c87fed845eb13823edef8a, size = 208.5 MB
metadata: 1440 files -> <repo>\data\metadata.csv
emotion
neutral       96
calm         192
happy        192
sad          192
angry        192
fearful      192
disgust      192
surprised    192
PASS data: exactly 1440 files | n=1440
PASS data: neutral=96, other emotions=192 | {'neutral': 96, 'calm': 192, 'happy': 192, 'sad': 192, 'angry': 192, 'fearful': 192, 'disgust': 192, 'surprised': 192}
PASS data: 60 files per actor (24 actors) | [60]
PASS data: 12 F / 12 M actors | {'M': 12, 'F': 12}
PASS data: label map fixed (index = code-1) and saved | {'neutral': 0, 'calm': 1, 'happy': 2, 'sad': 3, 'angry': 4, 'fearful': 5, 'disgust': 6, 'surprised': 7}

fold | actors (F/M)
  0  | 04F, 07M, 14F, 17M
  1  | 11M, 15M, 18

## Step 1b. Validation checklist (+ overfit a tiny batch)

In [4]:
run("checks", "src.checks", "--overfit")

<repo>\src\train.py:327: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  losses.append(float(loss))
Checks run 2026-11-21 17:34:17 (56 s)

| # | Check | Result | Detail |
|---|---|---|---|
| 1 | metrics (random): UAR==balanced_accuracy, macroF1==f1(macro), acc, mean(recalls)==UAR | PASS | /dUAR/=0.0e+00 /dF1/=0.0e+00 /dAcc/=0.0e+00 /dRec/=0.0e+00 |
| 2 | metrics (class never predicted): UAR==balanced_accuracy, macroF1==f1(macro), acc, mean(recalls)==UAR | PASS | /dUAR/=0.0e+00 /dF1/=0.0e+00 /dAcc/=0.0e+00 /dRec/=0.0e+00 |
| 3 | metrics (imbalanced): UAR==balanced_accuracy, macroF1==f1(macro), acc, mean(recalls)==UAR | PASS | /dUAR/=0.0e+00 /dF1/=0.0e+00 /dAcc/=0.0e+00 /dRec/=0.0e+00 |
| 4 | data: exactly 1440 files | PASS | n=1440 |
| 5 | data: neutral=96, ot

RuntimeError: checks failed -> <repo>\results\logs\checks.log

## Step 2. Preliminary runs (untuned defaults, 6 folds × seed 0) — M1, then M3, then M2

In [5]:
run("prelim_m1", "src.run_experiments", "--model", "m1", "--folds", "0-5", "--seeds", "0", "--tag", "prelim", "--skip-existing")

== m1_logmel_actor_f0_s0_prelim | cuda | params 596,232 (trainable 596,232) | cfg 65c4b81c8c
[m1_logmel_actor_f0_s0_prelim] ep   1 lr 1.00e-03 | train loss 1.990 acc 0.216 | val loss 1.932 UAR 0.242 *
[m1_logmel_actor_f0_s0_prelim] ep   2 lr 1.00e-03 | train loss 1.883 acc 0.302 | val loss 1.880 UAR 0.266 *
[m1_logmel_actor_f0_s0_prelim] ep   3 lr 1.00e-03 | train loss 1.864 acc 0.314 | val loss 1.845 UAR 0.309 *
[m1_logmel_actor_f0_s0_prelim] ep   4 lr 1.00e-03 | train loss 1.790 acc 0.345 | val loss 1.810 UAR 0.305
[m1_logmel_actor_f0_s0_prelim] ep   5 lr 1.00e-03 | train loss 1.730 acc 0.379 | val loss 1.752 UAR 0.359 *
[m1_logmel_actor_f0_s0_prelim] ep   6 lr 1.00e-03 | train loss 1.720 acc 0.393 | val loss 1.711 UAR 0.367 *
[m1_logmel_actor_f0_s0_prelim] ep   7 lr 1.00e-03 | train loss 1.663 acc 0.435 | val loss 1.670 UAR 0.398 *
[m1_logmel_actor_f0_s0_prelim] ep   8 lr 1.00e-03 | train loss 1.617 acc 0.453 | val loss 1.672 UAR 0.387
[m1_logmel_actor_f0_s0_prelim] ep   9 lr 1.00e-

RuntimeError: prelim_m1 failed -> <repo>\results\logs\prelim_m1.log

In [6]:
run("prelim_m3", "src.run_experiments", "--model", "m3", "--folds", "0-5", "--seeds", "0", "--tag", "prelim", "--skip-existing")

== m3_logmel_actor_f0_s0_prelim | cuda | params 2,423,048 (trainable 2,423,048) | cfg 97b7549749
[m3_logmel_actor_f0_s0_prelim] ep   1 lr 1.00e-04 | train loss 2.148 acc 0.131 | val loss 2.022 UAR 0.238 *
[m3_logmel_actor_f0_s0_prelim] ep   2 lr 2.00e-04 | train loss 2.023 acc 0.207 | val loss 1.987 UAR 0.316 *
[m3_logmel_actor_f0_s0_prelim] ep   3 lr 3.00e-04 | train loss 1.923 acc 0.291 | val loss 1.985 UAR 0.273
[m3_logmel_actor_f0_s0_prelim] ep   4 lr 4.00e-04 | train loss 1.880 acc 0.292 | val loss 1.818 UAR 0.289
[m3_logmel_actor_f0_s0_prelim] ep   5 lr 5.00e-04 | train loss 1.824 acc 0.322 | val loss 2.019 UAR 0.270
[m3_logmel_actor_f0_s0_prelim] ep   6 lr 5.00e-04 | train loss 1.787 acc 0.366 | val loss 1.729 UAR 0.375 *
[m3_logmel_actor_f0_s0_prelim] ep   7 lr 5.00e-04 | train loss 1.737 acc 0.386 | val loss 1.702 UAR 0.367
[m3_logmel_actor_f0_s0_prelim] ep   8 lr 5.00e-04 | train loss 1.714 acc 0.376 | val loss 1.801 UAR 0.324
[m3_logmel_actor_f0_s0_prelim] ep   9 lr 5.00e-04

In [7]:
run("prelim_m2", "src.run_experiments", "--model", "m2", "--folds", "0-5", "--seeds", "0", "--tag", "prelim", "--skip-existing")

== m2_logmel_actor_f0_s0_prelim | cuda | params 1,276,904 (trainable 1,276,904) | cfg e008d30dd9
[m2_logmel_actor_f0_s0_prelim] ep   1 lr 1.00e-03 | train loss 2.020 acc 0.198 | val loss 1.927 UAR 0.234 *
[m2_logmel_actor_f0_s0_prelim] ep   2 lr 1.00e-03 | train loss 1.883 acc 0.296 | val loss 1.855 UAR 0.297 *
[m2_logmel_actor_f0_s0_prelim] ep   3 lr 1.00e-03 | train loss 1.815 acc 0.330 | val loss 1.828 UAR 0.297
[m2_logmel_actor_f0_s0_prelim] ep   4 lr 1.00e-03 | train loss 1.776 acc 0.358 | val loss 1.784 UAR 0.316 *
[m2_logmel_actor_f0_s0_prelim] ep   5 lr 1.00e-03 | train loss 1.711 acc 0.392 | val loss 1.744 UAR 0.363 *
[m2_logmel_actor_f0_s0_prelim] ep   6 lr 1.00e-03 | train loss 1.664 acc 0.446 | val loss 1.657 UAR 0.430 *
[m2_logmel_actor_f0_s0_prelim] ep   7 lr 1.00e-03 | train loss 1.633 acc 0.423 | val loss 1.863 UAR 0.293
[m2_logmel_actor_f0_s0_prelim] ep   8 lr 1.00e-03 | train loss 1.580 acc 0.476 | val loss 1.770 UAR 0.328
[m2_logmel_actor_f0_s0_prelim] ep   9 lr 1.00

## Step 3. Optional extras for the progress report (only if time allows)

In [8]:
run("prelim_m1_random", "src.run_experiments", "--model", "m1", "--protocol", "random", "--seeds", "0", "--tag", "prelim", "--skip-existing")

== m1_logmel_random_fR_s0_prelim | cuda | params 596,232 (trainable 596,232) | cfg 65c4b81c8c
[m1_logmel_random_fR_s0_prelim] ep   1 lr 1.00e-03 | train loss 2.006 acc 0.208 | val loss 1.873 UAR 0.293 *
[m1_logmel_random_fR_s0_prelim] ep   2 lr 1.00e-03 | train loss 1.907 acc 0.270 | val loss 1.769 UAR 0.363 *
[m1_logmel_random_fR_s0_prelim] ep   3 lr 1.00e-03 | train loss 1.868 acc 0.289 | val loss 1.727 UAR 0.434 *
[m1_logmel_random_fR_s0_prelim] ep   4 lr 1.00e-03 | train loss 1.833 acc 0.316 | val loss 1.671 UAR 0.445 *
[m1_logmel_random_fR_s0_prelim] ep   5 lr 1.00e-03 | train loss 1.776 acc 0.344 | val loss 1.604 UAR 0.480 *
[m1_logmel_random_fR_s0_prelim] ep   6 lr 1.00e-03 | train loss 1.720 acc 0.385 | val loss 1.540 UAR 0.469
[m1_logmel_random_fR_s0_prelim] ep   7 lr 1.00e-03 | train loss 1.684 acc 0.405 | val loss 1.492 UAR 0.488 *
[m1_logmel_random_fR_s0_prelim] ep   8 lr 1.00e-03 | train loss 1.628 acc 0.460 | val loss 1.422 UAR 0.566 *
[m1_logmel_random_fR_s0_prelim] ep  

RuntimeError: prelim_m1_random failed -> <repo>\results\logs\prelim_m1_random.log

In [10]:
run("prelim_m4", "src.run_experiments", "--model", "m4", "--folds", "0", "--seeds", "0", "--tag", "prelim", "--skip-existing")

W1121 18:06:34.031000 57392 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
<venv>/site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in <home>\.cache\huggingface\hub\models--facebook--wav2vec2-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warn